# T3.1 — UNet segmentation on REAL AI4Shipwrecks masks [COLAB]

ResNet18-UNet (ImageNet-pretrained, `segmentation-models-pytorch`) on the **real** AI4Shipwrecks
shipwreck masks. Reports **IoU + F1 of the POSITIVE class only** (R3 — never pixel accuracy).
Benchmark: SOTA ~0.445 IoU, UNet ~0.411 (AI4Shipwrecks paper); target ≥ 0.40.

**Data:** put the `AI4Shipwrecks/` folder in your Drive — it has `train/{images,labels}` (141)
and `test/{images,labels}` (120) real shipwreck surveys + binary masks. The official train/test
split is already leak-free by wreck. Full surveys (~5579×1728) are tiled to 512² here.

The synthetic ghost_net YOLO-seg set is a SEPARATE synthetic-only result (R4) — not this number.
Run top-to-bottom on GPU.

In [ ]:
# 0. Setup
from google.colab import drive; drive.mount('/content/drive')
%cd /content
!git clone https://github.com/ASugandhan/SamudraPrahari.git || (cd SamudraPrahari && git pull)
%cd SamudraPrahari
!pip install -q -e '.[models]' segmentation-models-pytorch albumentations onnxconverter-common
import yaml; CFG = yaml.safe_load(open('configs/training.yaml'))['segmentation']; print(CFG)

In [ ]:
# 1. Locate AI4Shipwrecks in Drive; tile train/test surveys -> 512 crops (image+mask pairs)
import os, glob, re, random, cv2, numpy as np, subprocess
from pathlib import Path
from samudra.common.splits import split_by_group
DRIVE = '/content/drive/MyDrive'
CROP, STRIDE = CFG['crop'], CFG['stride']

if not Path('AI4Shipwrecks/train/images').exists():
    for c in glob.glob(f'{DRIVE}/**/AI4Shipwrecks', recursive=True):
        if Path(c, 'train/images').exists(): os.symlink(c, 'AI4Shipwrecks'); break
assert Path('AI4Shipwrecks/train/images').exists(), f'put AI4Shipwrecks/ in {DRIVE}'

def offsets(n, c, s):
    if n <= c: return [0]
    o = list(range(0, n - c + 1, s))
    if o[-1] != n - c: o.append(n - c)
    return o

def tile_split(img_dir, lbl_dir, out, keep_bg):
    Path(out, 'images').mkdir(parents=True, exist_ok=True); Path(out, 'labels').mkdir(parents=True, exist_ok=True)
    rng = random.Random(1337); kept = 0
    for ip in sorted(glob.glob(f'{img_dir}/*.png')):
        stem = Path(ip).stem; mp = f'{lbl_dir}/{stem}.png'
        if not Path(mp).exists(): continue
        im = cv2.imread(ip, cv2.IMREAD_GRAYSCALE); m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if im is None or m is None: continue
        H, W = im.shape
        for y in offsets(H, CROP, STRIDE):
            for x in offsets(W, CROP, STRIDE):
                ci = im[y:y+CROP, x:x+CROP]; cm = (m[y:y+CROP, x:x+CROP] > 0).astype(np.uint8) * 255
                if ci.shape != (CROP, CROP): continue
                if cm.max() == 0 and rng.random() > keep_bg: continue   # subsample pure-bg crops (train)
                fn = f'{stem}__{y}_{x}.png'
                cv2.imwrite(f'{out}/images/{fn}', ci); cv2.imwrite(f'{out}/labels/{fn}', cm); kept += 1
    return kept

# leak-free val carved from TRAIN by wreck (test/ is already a held-out wreck set)
train_imgs = sorted(glob.glob('AI4Shipwrecks/train/images/*.png'))
wreck = lambda p: re.sub(r'_\d+$', '', Path(p).stem)
assign = split_by_group([wreck(p) for p in train_imgs], seed=1337, ratios=(0.85, 0.15, 0.0))
val_wrecks = {w for w, s in assign.items() if s == 'val'}
print('val wrecks:', sorted(val_wrecks))

# stage train/val images into temp dirs by wreck, then tile
for split in ('train', 'val'):
    Path(f'_ai4_stage/{split}/images').mkdir(parents=True, exist_ok=True); Path(f'_ai4_stage/{split}/labels').mkdir(parents=True, exist_ok=True)
for ip in train_imgs:
    s = 'val' if wreck(ip) in val_wrecks else 'train'; stem = Path(ip).stem
    os.symlink(os.path.abspath(ip), f'_ai4_stage/{s}/images/{stem}.png')
    os.symlink(os.path.abspath(f'AI4Shipwrecks/train/labels/{stem}.png'), f'_ai4_stage/{s}/labels/{stem}.png')

# keep_bg=0.5: train must see plenty of pure-background crops or the model over-predicts and
# floods the test (which has ALL background crops) with false positives -> low IoU. (v1 used
# 0.15 and got IoU ~0.28 with FP >> TP; matching the train/test background rate fixes that.)
n_tr = tile_split('_ai4_stage/train/images', '_ai4_stage/train/labels', 'unet_data/train', keep_bg=0.5)
n_va = tile_split('_ai4_stage/val/images', '_ai4_stage/val/labels', 'unet_data/val', keep_bg=1.0)
n_te = tile_split('AI4Shipwrecks/test/images', 'AI4Shipwrecks/test/labels', 'unet_data/test', keep_bg=1.0)
print(f'crops -> train {n_tr} | val {n_va} | test {n_te}')

In [ ]:
# 2. Dataset (grayscale->3ch, ImageNet norm for the pretrained encoder) + sonar-safe aug
import torch
from torch.utils.data import Dataset, DataLoader
MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)

class SegDS(Dataset):
    def __init__(self, root, augment):
        self.imgs = sorted(glob.glob(f'{root}/images/*.png')); self.root = root; self.augment = augment
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        ip = self.imgs[i]; fn = Path(ip).name
        im = cv2.imread(ip, cv2.IMREAD_GRAYSCALE); m = cv2.imread(f'{self.root}/labels/{fn}', cv2.IMREAD_GRAYSCALE)
        x = np.repeat(im[..., None], 3, 2).astype(np.float32) / 255.0
        y = (m > 0).astype(np.float32)
        if self.augment:
            if random.random() < 0.5: x, y = x[:, ::-1], y[:, ::-1]
            if random.random() < 0.5: x, y = x[::-1], y[::-1]
            k = random.randint(0, 3); x, y = np.rot90(x, k), np.rot90(y, k)
        x = (x - MEAN) / STD
        return torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1))), torch.from_numpy(np.ascontiguousarray(y))[None]

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
tr = DataLoader(SegDS('unet_data/train', True), batch_size=CFG['batch'], shuffle=True, drop_last=True, num_workers=2)
va = DataLoader(SegDS('unet_data/val', False), batch_size=CFG['batch'], num_workers=2)
print('device', DEVICE, '| train batches', len(tr), '| val batches', len(va))

In [ ]:
# 3. Model + Dice+BCE loss + train (monitor positive-class IoU on the leak-free val)
import segmentation_models_pytorch as smp
import torch.nn as nn
model = smp.Unet(CFG['encoder'], encoder_weights='imagenet', in_channels=3, classes=1).to(DEVICE)
dice = smp.losses.DiceLoss(mode='binary'); bce = nn.BCEWithLogitsLoss()
def loss_fn(logits, y): return dice(logits, y) + bce(logits, y)
opt = torch.optim.Adam(model.parameters(), lr=CFG['lr'])

@torch.no_grad()
def iou_f1(loader):
    model.eval(); tp = fp = fn = 0
    for x, y in loader:
        p = (torch.sigmoid(model(x.to(DEVICE))) > 0.5).cpu().numpy().astype(bool)
        g = y.numpy().astype(bool)
        tp += np.logical_and(p, g).sum(); fp += np.logical_and(p, ~g).sum(); fn += np.logical_and(~p, g).sum()
    iou = tp / (tp + fp + fn + 1e-9); f1 = 2 * tp / (2 * tp + fp + fn + 1e-9)
    return float(iou), float(f1)

best = 0.0
for ep in range(CFG['epochs']):
    model.train()
    for x, y in tr:
        opt.zero_grad(); loss = loss_fn(model(x.to(DEVICE)), y.to(DEVICE)); loss.backward(); opt.step()
    if ep % 5 == 0 or ep == CFG['epochs'] - 1:
        iou, f1 = iou_f1(va); best = max(best, iou)
        print(f'ep {ep:3d}  val IoU(+) {iou:.3f}  F1(+) {f1:.3f}')
print('best val IoU(+):', round(best, 3))

In [ ]:
# 4. FINAL eval on the official REAL test split — IoU + F1 of the POSITIVE class (R3)
te = DataLoader(SegDS('unet_data/test', False), batch_size=CFG['batch'], num_workers=2)
test_iou, test_f1 = iou_f1(te)
print(f'AI4Shipwrecks TEST (real): IoU(+) {test_iou:.3f}  F1(+) {test_f1:.3f}   '
      f'(benchmark SOTA ~0.445 / UNet ~0.411; target >=0.40)')
import json; Path('out').mkdir(exist_ok=True)
json.dump({'task': 'T3.1', 'model': 'unet_resnet18', 'data': 'AI4Shipwrecks real (test split)',
           'test_IoU_positive': round(test_iou, 4), 'test_F1_positive': round(test_f1, 4),
           'metric_note': 'positive-class IoU/F1 only, NOT pixel accuracy (R3)'},
          open('out/metrics_t31.json', 'w'), indent=2)

In [ ]:
# 5. Export ONNX (<=50 MB, single self-contained file) + register
import onnx, os
model.eval().cpu()
torch.onnx.export(model, torch.randn(1, 3, CFG['crop'], CFG['crop']), 'models/stage_b_unet.onnx',
                  opset_version=13, input_names=['input'], output_names=['logits'],
                  dynamic_axes={'input': {0: 'b'}, 'logits': {0: 'b'}})
m = onnx.load('models/stage_b_unet.onnx'); m.ir_version = min(m.ir_version, 10)
onnx.save_model(m, 'models/stage_b_unet.onnx', save_as_external_data=False)
if os.path.exists('models/stage_b_unet.onnx.data'): os.remove('models/stage_b_unet.onnx.data')
# fp16 for edge (R11 <=50 MB) — resnet18-UNet fp32 is ~57 MB; fp16 ~29 MB, IoU unchanged
from onnxconverter_common import float16
onnx.save(float16.convert_float_to_float16(onnx.load('models/stage_b_unet.onnx'), keep_io_types=True),
          'models/stage_b_unet.onnx')
print('ONNX size:', round(os.path.getsize('models/stage_b_unet.onnx') / 1e6, 2), 'MB (fp16, <=50)')
!python scripts/register_model.py --onnx models/stage_b_unet.onnx --name unet --trained-on 'AI4Shipwrecks real masks (train)' --metrics out/metrics_t31.json
!cp models/stage_b_unet.onnx models/registry.json out/metrics_t31.json /content/drive/MyDrive/ 2>/dev/null; echo done

## After it runs
1. Note the **TEST IoU(+)** vs the benchmark (target ≥0.40; SOTA ~0.445).
2. Copy back `models/stage_b_unet.onnx`, `models/registry.json`, `out/metrics_t31.json`. Commit + push.
3. Locally verify (CPU, onnxruntime — no torch/smp needed):
   `python scripts/eval/unet_eval.py --split test --real-only`

**R3:** only positive-class IoU/F1 are reported — never pixel accuracy (99.9% of pixels are seabed).
**R4:** this is the REAL number; the synthetic ghost_net seg is reported separately if run.